# Prompt & Pipeline Data Audit — Visual Deep-Dive

Loads JSONL pipeline outputs from `data/atoms/` and produces charts covering:
1. Noise analysis
2. Completion funnel
3. Prompt type effectiveness
4. Size vs completion
5. Session heatmap
6. Session length distribution
7. Context switching
8. Jaccard distribution
9. Link quality
10. Recommendations dashboard
11. Ghost plans

In [ ]:
# Setup: load JSONL files into DataFrames
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.style.use("seaborn-v0_8-whitegrid")

ATOMS_DIR = Path.home() / "Workspace/meta-organvm/organvm-corpvs-testamentvm/data/atoms"

def load_jsonl(name):
    path = ATOMS_DIR / name
    if not path.exists():
        print(f"WARNING: {path} not found")
        return pd.DataFrame()
    records = []
    with path.open() as f:
        for line in f:
            s = line.strip()
            if s:
                records.append(json.loads(s))
    return pd.json_normalize(records, sep="_")

prompts_df = load_jsonl("annotated-prompts.jsonl")
tasks_df = load_jsonl("atomized-tasks.jsonl")
links_df = load_jsonl("atom-links.jsonl")

print(f"Prompts: {len(prompts_df):,}")
print(f"Tasks:   {len(tasks_df):,}")
print(f"Links:   {len(links_df):,}")

In [ ]:
# Cell 2: Noise Analysis
from organvm_engine.prompts.audit import audit_noise

noise_results = audit_noise(prompts_df.to_dict("records"))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of noise types
noise_types = noise_results["noise_by_type"]
if noise_types:
    ax1.barh(list(noise_types.keys()), list(noise_types.values()), color="#e74c3c")
    ax1.set_xlabel("Count")
    ax1.set_title("Noise by Type")
    ax1.invert_yaxis()

# Before/after pie
labels = ["Signal", "Noise"]
sizes = [noise_results["signal_count"], noise_results["noise_count"]]
colors = ["#2ecc71", "#e74c3c"]
ax2.pie(sizes, labels=labels, colors=colors, autopct="%1.1f%%", startangle=90)
ax2.set_title(f"Signal vs Noise ({noise_results['total']:,} total)")

plt.tight_layout()
plt.show()

print(f"Signal: {noise_results['signal_count']:,} | Noise: {noise_results['noise_count']:,} "
      f"({noise_results['noise_pct']}%)")

In [ ]:
# Cell 3: Completion Funnel — stacked bar by organ
from organvm_engine.prompts.audit import audit_completion

comp = audit_completion(
    tasks_df.to_dict("records"),
    prompts_df.to_dict("records"),
    links_df.to_dict("records"),
)

by_organ = comp["by_organ"]
if by_organ:
    organs = sorted(by_organ.keys())
    total = [by_organ[o]["total"] for o in organs]
    hq_linked = [by_organ[o]["hq_linked"] for o in organs]
    completed = [by_organ[o]["completed"] for o in organs]

    x = range(len(organs))
    width = 0.25
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar([i - width for i in x], total, width, label="Total Tasks", color="#3498db")
    ax.bar(x, hq_linked, width, label="HQ Linked (J>=0.30)", color="#f39c12")
    ax.bar([i + width for i in x], completed, width, label="Completed", color="#2ecc71")
    ax.set_xticks(x)
    ax.set_xticklabels(organs, rotation=45, ha="right")
    ax.set_ylabel("Count")
    ax.set_title("Completion Funnel by Organ")
    ax.legend()
    plt.tight_layout()
    plt.show()

funnel = comp["funnel_summary"]
print(f"Plans: {funnel['plans_parsed']} -> Tasks: {funnel['total_tasks']} "
      f"-> HQ Linked: {funnel['tasks_with_hq_links']} -> Completed: {funnel['completed_tasks']}")
print(f"Completion rate: {funnel['completion_rate']}% | Linkage rate: {funnel['linkage_rate']}%")

In [ ]:
# Cell 4: Prompt Type Effectiveness
from organvm_engine.prompts.audit import audit_effectiveness

eff = audit_effectiveness(
    prompts_df.to_dict("records"),
    tasks_df.to_dict("records"),
    links_df.to_dict("records"),
)

by_type = eff["by_type"]
if by_type:
    types = sorted(by_type.keys(), key=lambda t: by_type[t]["prompts"], reverse=True)
    prompts_count = [by_type[t]["prompts"] for t in types]
    linked = [by_type[t]["linked_tasks"] for t in types]
    completed = [by_type[t]["completed_tasks"] for t in types]

    x = range(len(types))
    width = 0.25
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar([i - width for i in x], prompts_count, width, label="Prompts", color="#3498db")
    ax.bar(x, linked, width, label="Linked Tasks", color="#f39c12")
    ax.bar([i + width for i in x], completed, width, label="Completed Tasks", color="#2ecc71")
    ax.set_xticks(x)
    ax.set_xticklabels(types, rotation=45, ha="right")
    ax.set_ylabel("Count")
    ax.set_title("Prompt Type Effectiveness")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 5: Size vs Completion — box plot
by_size = eff["by_size"]
if by_size:
    size_order = ["terse", "short", "medium", "long"]
    sizes = [s for s in size_order if s in by_size]
    linked_vals = [by_size[s]["linked_tasks"] for s in sizes]
    completed_vals = [by_size[s]["completed_tasks"] for s in sizes]

    fig, ax = plt.subplots(figsize=(8, 5))
    x = range(len(sizes))
    width = 0.35
    ax.bar([i - width/2 for i in x], linked_vals, width, label="Linked Tasks", color="#f39c12")
    ax.bar([i + width/2 for i in x], completed_vals, width, label="Completed", color="#2ecc71")
    ax.set_xticks(x)
    ax.set_xticklabels(sizes)
    ax.set_xlabel("Prompt Size Class")
    ax.set_ylabel("Task Count")
    ax.set_title("Prompt Size vs Task Outcomes")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 6: Session Heatmap — hour-of-day x day-of-week

from organvm_engine.prompts.audit import audit_sessions

sess = audit_sessions(prompts_df.to_dict("records"))

# Build heatmap from raw timestamps
if "source_timestamp" in prompts_df.columns:
    ts_series = pd.to_datetime(prompts_df["source_timestamp"], errors="coerce", utc=True)
    valid = ts_series.dropna()
    if len(valid) > 0:
        hours = valid.dt.hour
        days = valid.dt.day_name()
        day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

        heatmap_data = pd.crosstab(
            pd.Categorical(days, categories=day_order, ordered=True),
            hours,
        ).reindex(columns=range(24), fill_value=0)

        fig, ax = plt.subplots(figsize=(14, 5))
        im = ax.imshow(heatmap_data.values, aspect="auto", cmap="YlOrRd")
        ax.set_xticks(range(24))
        ax.set_xticklabels([f"{h:02d}" for h in range(24)])
        ax.set_yticks(range(len(day_order)))
        ax.set_yticklabels(day_order)
        ax.set_xlabel("Hour (UTC)")
        ax.set_ylabel("Day of Week")
        ax.set_title("Prompt Activity Heatmap")
        plt.colorbar(im, ax=ax, label="Prompt Count")
        plt.tight_layout()
        plt.show()
else:
    print("No timestamp column found — skipping heatmap")

In [ ]:
# Cell 7: Session Length Distribution
length_dist = sess["length_dist"]
if length_dist:
    fig, ax = plt.subplots(figsize=(8, 5))
    buckets = list(length_dist.keys())
    counts = list(length_dist.values())
    ax.bar(buckets, counts, color="#9b59b6")
    ax.set_xlabel("Prompts per Session")
    ax.set_ylabel("Session Count")
    ax.set_title(f"Session Length Distribution ({sess['total_sessions']} sessions)")
    for i, v in enumerate(counts):
        if v > 0:
            ax.text(i, v + 0.5, str(v), ha="center", fontsize=9)
    plt.tight_layout()
    plt.show()

churn = sess["churn"]
print(f"Single-prompt sessions: {churn['single_prompt_sessions']} "
      f"({churn['churn_ratio']}% churn)")

In [ ]:
# Cell 8: Context Switching — projects per day timeline
if "source_timestamp" in prompts_df.columns and "source_project_slug" in prompts_df.columns:
    df = prompts_df[["source_timestamp", "source_project_slug"]].copy()
    df["date"] = pd.to_datetime(df["source_timestamp"], errors="coerce", utc=True).dt.date
    df = df.dropna(subset=["date"])

    daily_projects = df.groupby("date")["source_project_slug"].nunique()

    if len(daily_projects) > 0:
        fig, ax = plt.subplots(figsize=(14, 4))
        ax.fill_between(daily_projects.index, daily_projects.values, alpha=0.3, color="#e74c3c")
        ax.plot(daily_projects.index, daily_projects.values, color="#e74c3c", linewidth=1)
        ax.set_xlabel("Date")
        ax.set_ylabel("Distinct Projects")
        ax.set_title("Context Switching: Projects per Day")
        ax.axhline(y=daily_projects.mean(), color="#2c3e50", linestyle="--",
                   label=f"Avg: {daily_projects.mean():.1f}")
        ax.legend()
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("Missing columns for context switch analysis")

In [ ]:
# Cell 9: Jaccard Distribution with threshold lines
from organvm_engine.prompts.audit import audit_links

links_audit = audit_links(
    links_df.to_dict("records"),
    tasks_df.to_dict("records"),
    prompts_df.to_dict("records"),
)

if "jaccard" in links_df.columns and len(links_df) > 0:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.hist(links_df["jaccard"], bins=50, color="#3498db", alpha=0.7, edgecolor="white")

    for thresh, color, label in [
        (0.15, "#e74c3c", "Current (0.15)"),
        (0.30, "#f39c12", "Recommended (0.30)"),
        (0.50, "#2ecc71", "Strict (0.50)"),
    ]:
        ax.axvline(x=thresh, color=color, linestyle="--", linewidth=2, label=label)

    ax.set_xlabel("Jaccard Similarity")
    ax.set_ylabel("Link Count")
    ax.set_title("Jaccard Score Distribution")
    ax.legend()
    plt.tight_layout()
    plt.show()

    # Threshold analysis table
    ta = links_audit["threshold_analysis"]
    print(f"{'Threshold':<12} {'Links':>8} {'Tasks':>8} {'% Total':>10}")
    print("-" * 40)
    for t in sorted(ta.keys()):
        info = ta[t]
        print(f">= {t:<9} {info['links']:>8,} {info['tasks_with_links']:>8,} "
              f"{info['pct_of_total']:>9.1f}%")

In [ ]:
# Cell 10: Link Quality — fan-out scatter

if "task_id" in links_df.columns and len(links_df) > 0:
    fanout = links_df.groupby("task_id").agg(
        link_count=("jaccard", "count"),
        avg_jaccard=("jaccard", "mean"),
    ).reset_index()

    fig, ax = plt.subplots(figsize=(10, 6))
    scatter = ax.scatter(
        fanout["link_count"], fanout["avg_jaccard"],
        alpha=0.4, s=10, c=fanout["link_count"], cmap="YlOrRd",
    )
    ax.axvline(x=100, color="#e74c3c", linestyle="--", alpha=0.7, label="Fan-out threshold (100)")
    ax.set_xlabel("Links per Task (fan-out)")
    ax.set_ylabel("Average Jaccard")
    ax.set_title("Task Fan-out vs Link Quality")
    ax.set_xscale("log")
    ax.legend()
    plt.colorbar(scatter, ax=ax, label="Link Count")
    plt.tight_layout()
    plt.show()

    print(f"Empty-FP contamination: {links_audit['empty_fp_count']:,} "
          f"({links_audit['empty_fp_pct']}%)")
    print(f"Generic-tag-only links: {links_audit['generic_tag_links']:,} "
          f"({links_audit['generic_tag_pct']}%)")
    print(f"High fan-out tasks (>100 links): {links_audit['high_fanout_count']}")

In [ ]:
# Cell 11: Recommendations Dashboard
from organvm_engine.prompts.audit import generate_recommendations

recs = generate_recommendations(noise_results, comp, eff, sess, links_audit)

if recs:
    rec_df = pd.DataFrame(recs)
    # Color-code by priority
    colors = {"P0": "#e74c3c", "P1": "#f39c12", "P2": "#3498db"}

    fig, ax = plt.subplots(figsize=(12, max(3, len(recs) * 0.6)))
    ax.axis("off")
    table = ax.table(
        cellText=rec_df[["priority", "category", "finding", "recommendation"]].values,
        colLabels=["Priority", "Category", "Finding", "Recommendation"],
        loc="center",
        cellLoc="left",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.auto_set_column_width(col=list(range(4)))

    # Color priority cells
    for i in range(len(recs)):
        cell = table[i + 1, 0]
        cell.set_facecolor(colors.get(recs[i]["priority"], "#bdc3c7"))
        cell.set_text_props(color="white", fontweight="bold")

    ax.set_title(f"Recommendations ({len(recs)} items)", fontsize=14, fontweight="bold", pad=20)
    plt.tight_layout()
    plt.show()
else:
    print("No recommendations generated — pipeline is healthy!")

In [ ]:
# Cell 12: Ghost Plans — highest-effort plans with 0 completion
ghosts = comp.get("ghost_plans", [])
if ghosts:
    ghost_df = pd.DataFrame(ghosts[:25])
    ghost_df["plan_short"] = ghost_df["plan"].apply(
        lambda p: p.rsplit("/", 1)[-1] if "/" in p else p,
    )

    fig, ax = plt.subplots(figsize=(12, max(4, len(ghost_df) * 0.35)))
    ax.barh(ghost_df["plan_short"], ghost_df["tasks"], color="#e74c3c", alpha=0.8)
    ax.set_xlabel("Tasks in Plan")
    ax.set_title(f"Ghost Plans: {comp['ghost_plan_count']} plans with 0 completion & 0 HQ links")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("No ghost plans detected")